In [ ]:
import pandas as pd

player_raw = pd.read_csv("2022-2023 Football Player Stats.csv", delimiter=';', encoding='latin1')

pd.set_option('display.max_columns', None)
print(player_raw.head())


   Rk             Player Nation   Pos         Squad            Comp  Age  \
0   1   Brenden Aaronson    USA  MFFW  Leeds United  Premier League   22   
1   2   Yunis Abdelhamid    MAR    DF         Reims         Ligue 1   35   
2   3      Himad Abdelli    FRA  MFFW        Angers         Ligue 1   23   
3   4  Salis Abdul Samed    GHA    MF          Lens         Ligue 1   22   
4   5    Laurent Abergel    FRA    MF       Lorient         Ligue 1   30   

   Born  MP  Starts   Min   90s  Goals  Shots   SoT  SoT%  G/Sh  G/SoT  \
0  2000  20      19  1596  17.7      1   1.53  0.28  18.5  0.04   0.20   
1  1987  22      22  1980  22.0      0   0.86  0.05   5.3  0.00   0.00   
2  1999  14       8   770   8.6      0   1.05  0.35  33.3  0.00   0.00   
3  2000  20      20  1799  20.0      1   0.60  0.15  25.0  0.08   0.33   
4  1993  15      15  1165  12.9      0   0.31  0.00   0.0  0.00   0.00   

   ShoDist  ShoFK  ShoPK  PKatt  PasTotCmp  PasTotAtt  PasTotCmp%  PasTotDist  \
0     19.0   0.11

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Your list of player names
# player_list = ["Brenden Aaronson", "Yunis Abdelhamid", "Himad Abdelli"]
player_list = player_raw['Player'].unique()
# player_list = ['Yunis Abdelhamid']

# Base URL for player search
base_url = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query="

# DataFrame to store results
results = []

# Total number of players
total_players = len(player_list)

# Iterate through the player list with a counter
for i, player in enumerate(player_list, start=1):
    # Format the player name for the URL
    search_url = base_url + player.replace(" ", "+")
    print(f"Processing player {i}/{total_players}: {player}")

    try:
        # Fetch the page
        response = requests.get(search_url, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract the market value
        market_value_tag = soup.find("td", class_="rechts hauptlink")  # Target the specific class
        if market_value_tag:
            market_value = market_value_tag.text.strip()
        else:
            market_value = "Not Found"

        # Append the result
        results.append({"Player": player, "Market Value": market_value})
    except Exception as e:
        results.append({"Player": player, "Market Value": "Error: " + str(e)})

    # Pause to avoid being blocked by the website
    time.sleep(2)

# Convert results to DataFrame
df_results = pd.DataFrame(results)

# Display or save the DataFrame
#print(df_results)
df_results.to_csv("player_market_values.csv", index=False)
print("scrapping completed")
# Website: https://www.transfermarkt.com/

Processing player 1/2530: Brenden Aaronson
Processing player 2/2530: Yunis Abdelhamid
Processing player 3/2530: Himad Abdelli
Processing player 4/2530: Salis Abdul Samed
Processing player 5/2530: Laurent Abergel
Processing player 6/2530: Oliver Abildgaard
Processing player 7/2530: Matthis Abline
Processing player 8/2530: Abner
Processing player 9/2530: Zakaria Aboukhlal
Processing player 10/2530: Tammy Abraham
Processing player 11/2530: Francesco Acerbi
Processing player 12/2530: Mohamed Achi
Processing player 13/2530: Marcos Acuña
Processing player 14/2530: Che Adams
Processing player 15/2530: Tyler Adams
Processing player 16/2530: Sargis Adamyan
Processing player 17/2530: Tosin Adarabioyo
Processing player 18/2530: Martin Adeline
Processing player 19/2530: Karim Adeyemi
Processing player 20/2530: Amine Adli
Processing player 21/2530: Yacine Adli
Processing player 22/2530: Michel Aebischer
Processing player 23/2530: Felix Afena-Gyan
Processing player 24/2530: Emmanuel Agbadou
Processi

In [ ]:
# Loading the data back in
df_results_2 = pd.read_csv("player_market_values.csv")

# Removing the data that we could not pull Market Value
df_results_2 = df_results_2[df_results_2['Market Value'] != 'Not Found']
df_results_2 = df_results_2[df_results_2['Market Value'] != '-']

# Converting the Ks and Ms to actual Numbers

# Conversion function
def convert_market_value(value):
    value = value.replace("€", "").replace(",", "").strip()  # Remove currency symbol and commas
    if "m" in value:
        return float(value.replace("m", "")) * 1_000_000  # Convert millions to numeric
    elif "k" in value:
        return float(value.replace("k", "")) * 1_000  # Convert thousands to numeric
    else:
        return float(value)  # Return the number as-is if no unit

# Apply the conversion
df_results_2["Market Value Euros"] = df_results_2["Market Value"].apply(convert_market_value)

# Joining the Market Value to the Main DF
print(player_raw.shape)
player_raw = player_raw.merge(df_results_2, how = 'left', on = 'Player')
print(player_raw.shape)

# filtering out players that we don't have market value
player_raw = player_raw[~player_raw['Market Value Euros'].isna()]
print(player_raw.shape)
player_raw.head()

(2689, 124)
(2689, 126)
(2415, 126)


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,90s,Goals,Shots,SoT,SoT%,G/Sh,G/SoT,ShoDist,ShoFK,ShoPK,PKatt,PasTotCmp,PasTotAtt,PasTotCmp%,PasTotDist,PasTotPrgDist,PasShoCmp,PasShoAtt,PasShoCmp%,PasMedCmp,PasMedAtt,PasMedCmp%,PasLonCmp,PasLonAtt,PasLonCmp%,Assists,PasAss,Pas3rd,PPA,CrsPA,PasProg,PasAtt,PasLive,PasDead,PasFK,TB,Sw,PasCrs,TI,CK,CkIn,CkOut,CkStr,PasCmp,PasOff,PasBlocks,SCA,ScaPassLive,ScaPassDead,ScaDrib,ScaSh,ScaFld,ScaDef,GCA,GcaPassLive,GcaPassDead,GcaDrib,GcaSh,GcaFld,GcaDef,Tkl,TklWon,TklDef3rd,TklMid3rd,TklAtt3rd,TklDri,TklDriAtt,TklDri%,TklDriPast,Blocks,BlkSh,BlkPass,Int,Tkl+Int,Clr,Err,Touches,TouDefPen,TouDef3rd,TouMid3rd,TouAtt3rd,TouAttPen,TouLive,ToAtt,ToSuc,ToSuc%,ToTkl,ToTkl%,Carries,CarTotDist,CarPrgDist,CarProg,Car3rd,CPA,CarMis,CarDis,Rec,RecProg,CrdY,CrdR,2CrdY,Fls,Fld,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%,Market Value,Market Value Euros
0,1,Brenden Aaronson,USA,MFFW,Leeds United,Premier League,22,2000,20,19,1596,17.7,1,1.53,0.28,18.5,0.04,0.20,19.0,0.11,0.0,0.0,23.2,31.0,74.9,293.0,85.7,13.3,16.2,81.9,5.93,7.74,76.6,0.90,2.37,38.1,0.11,1.75,1.75,0.45,0.11,3.22,31.0,28.1,2.88,0.96,0.17,0.00,2.54,0.23,1.47,0.68,0.62,0.06,23.2,0.00,0.85,3.62,2.37,0.56,0.23,0.11,0.28,0.06,0.28,0.17,0.0,0.0,0.06,0.0,0.06,1.64,0.51,0.45,0.90,0.28,0.51,1.47,34.6,0.96,1.69,0.11,1.58,0.06,1.69,0.28,0.06,44.0,0.40,4.35,19.0,21.50,2.49,44.0,3.73,1.19,31.8,1.75,47.0,26.7,136.1,56.6,1.53,1.07,0.40,2.60,3.11,30.2,5.65,0.11,0.0,0.0,0.62,2.26,0.17,2.54,0.51,0.0,0.0,0.00,4.86,0.34,1.19,22.2,€15.00m,15000000.0
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,35,1987,22,22,1980,22.0,0,0.86,0.05,5.3,0.00,0.00,13.5,0.00,0.0,0.0,38.5,47.2,81.5,751.5,318.5,10.9,12.9,84.5,23.20,25.70,90.1,3.77,7.00,53.9,0.05,0.27,2.91,0.09,0.00,4.50,47.2,43.3,3.73,3.32,0.00,0.55,0.18,0.09,0.00,0.00,0.00,0.00,38.5,0.23,0.59,1.14,0.86,0.00,0.00,0.18,0.00,0.09,0.18,0.14,0.0,0.0,0.05,0.0,0.00,2.50,1.59,1.45,1.00,0.05,1.32,1.68,78.4,0.36,2.23,0.77,1.45,2.00,4.50,2.91,0.05,59.2,6.23,27.50,29.5,2.73,1.09,59.2,0.68,0.32,46.7,0.36,53.3,40.0,234.2,125.0,0.55,0.23,0.09,0.73,0.68,34.5,0.23,0.09,0.0,0.0,1.32,0.50,0.05,0.18,1.59,0.0,0.0,0.00,6.64,2.18,1.23,64.0,€400k,400000.0
2,3,Himad Abdelli,FRA,MFFW,Angers,Ligue 1,23,1999,14,8,770,8.6,0,1.05,0.35,33.3,0.00,0.00,19.2,0.00,0.0,0.0,40.0,49.5,80.8,676.0,188.1,18.5,22.0,84.1,15.50,18.70,82.6,4.42,5.93,74.5,0.00,1.51,3.95,1.74,0.35,6.40,49.5,48.1,1.16,0.35,0.12,0.23,1.05,0.81,0.00,0.00,0.00,0.00,40.0,0.23,1.16,2.67,2.44,0.00,0.00,0.12,0.00,0.12,0.00,0.00,0.0,0.0,0.00,0.0,0.00,2.91,1.40,1.28,1.40,0.23,1.63,2.67,60.9,1.05,1.51,0.12,1.40,0.93,3.84,0.93,0.00,62.6,0.93,11.40,36.0,17.40,1.16,62.6,3.84,2.09,54.5,1.51,39.4,48.5,298.5,151.0,2.56,2.56,0.47,2.09,1.05,43.4,5.93,0.12,0.0,0.0,1.74,1.28,0.00,1.05,1.40,0.0,0.0,0.00,8.14,0.93,1.05,47.1,€7.00m,7000000.0
3,4,Salis Abdul Samed,GHA,MF,Lens,Ligue 1,22,2000,20,20,1799,20.0,1,0.60,0.15,25.0,0.08,0.33,20.3,0.00,0.0,0.0,59.5,64.9,91.6,946.3,226.9,29.6,31.8,93.2,24.70,26.20,94.3,3.35,4.30,77.9,0.00,0.50,6.00,0.55,0.10,5.60,64.9,63.1,1.40,1.30,0.05,0.20,0.35,0.10,0.00,0.00,0.00,0.00,59.5,0.35,0.40,1.60,1.35,0.00,0.10,0.10,0.00,0.05,0.00,0.00,0.0,0.0,0.00,0.0,0.00,1.50,0.80,0.55,0.85,0.10,0.85,1.30,65.4,0.45,1.30,0.35,0.95,1.10,2.60,0.80,0.00,73.3,2.10,12.00,49.6,12.20,0.70,73.3,1.25,0.70,56.0,0.40,32.0,61.0,316.9,117.5,1.25,1.95,0.15,1.35,1.30,56.5,1.70,0.15,0.0,0.0,2.45,1.35,0.00,0.35,0.80,0.0,0.0,0.05,6.60,0.50,0.50,50.0,€5.00m,5000000.0
4,5,Laurent Abergel,FRA,MF,Lorient,Ligue 1,30,1993,15,15,1165,12.9,0,0.31,0.00,0.0,0.00,0.00,23.9,0.00,0.0,0.0,37.9,43.4,87.3,613.6,224.7,17.9,19.4,92.4,15.70,17.80,88.6,2.64,3.95,66.7,0.08,0.62,3.88,0.39,0.00,5.04,43.4,42.6,0.78,0.78,0.39,0.16,0.23,0.00,0.00,0.00,0.00,0.00,37.9,0.08,0.31,1.24,1.01,0.08,0.00,0.08,0.00,0.08,0.08,0.08,0.0,0.0,0.00,0.0,0.00,3.80,2.02,2.64,1.16,0.00,1.32,3.26,40.5,1.94,1.40,0.23,1.16,1.16,4.96,1.55,0.00,54.7,3.26,19.20,31.4,4.88,0.23,54.7,0.93,0.54,58.3,0.31,33.3,41.0,174.3,72.7,0.47,0.93,0.

In [ ]:
import pandas as pd

# Load the scraped market value data
df_results_2 = pd.read_csv("player_market_values.csv")

# Preview columns
print("Columns in scraped data:", df_results_2.columns.tolist())

# Clean invalid entries
df_results_2 = df_results_2[
    (df_results_2['Market Value'].notna()) &
    (~df_results_2['Market Value'].isin(['Not Found', '-']))
]

# Conversion function (robust)
def convert_market_value(value):
    try:
        value = str(value).replace("€", "").replace(",", "").strip().lower()
        if "m" in value:
            return float(value.replace("m", "")) * 1_000_000
        elif "k" in value:
            return float(value.replace("k", "")) * 1_000
        else:
            return float(value)
    except:
        return None  # fallback if conversion fails

# Apply conversion and verify
df_results_2["Market Value Euros"] = df_results_2["Market Value"].apply(convert_market_value)

print("✅ Conversion done. Non-null count:", df_results_2["Market Value Euros"].notna().sum())

# Merge with player_raw
print("Before merge:", player_raw.shape)
player_raw = player_raw.merge(df_results_2, how="left", on="Player")
print("After merge:", player_raw.shape)

# Check if the column exists after merge
if "Market Value Euros" in player_raw.columns:
    player_raw = player_raw[~player_raw["Market Value Euros"].isna()]
    print("After filtering:", player_raw.shape)
else:
    print("⚠️ 'Market Value Euros' column missing after merge!")

# Save cleaned dataset
player_raw.to_csv("player_data_with_market_value.csv", index=False)
print("✅ Saved: player_data_with_market_value.csv")


Columns in scraped data: ['Player', 'Market Value']
✅ Conversion done. Non-null count: 2266
Before merge: (2415, 128)
After merge: (2415, 130)
After filtering: (2415, 130)
✅ Saved: player_data_with_market_value.csv


In [ ]:
import pandas as pd
player_data = pd.read_csv("player_data_with_reddit_sentiment.csv")
player_data.columns.tolist()

['Rk',
 'Player',
 'Nation',
 'Pos',
 'Squad',
 'Comp',
 'Age',
 'Born',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Goals',
 'Shots',
 'SoT',
 'SoT%',
 'G/Sh',
 'G/SoT',
 'ShoDist',
 'ShoFK',
 'ShoPK',
 'PKatt',
 'PasTotCmp',
 'PasTotAtt',
 'PasTotCmp%',
 'PasTotDist',
 'PasTotPrgDist',
 'PasShoCmp',
 'PasShoAtt',
 'PasShoCmp%',
 'PasMedCmp',
 'PasMedAtt',
 'PasMedCmp%',
 'PasLonCmp',
 'PasLonAtt',
 'PasLonCmp%',
 'Assists',
 'PasAss',
 'Pas3rd',
 'PPA',
 'CrsPA',
 'PasProg',
 'PasAtt',
 'PasLive',
 'PasDead',
 'PasFK',
 'TB',
 'Sw',
 'PasCrs',
 'TI',
 'CK',
 'CkIn',
 'CkOut',
 'CkStr',
 'PasCmp',
 'PasOff',
 'PasBlocks',
 'SCA',
 'ScaPassLive',
 'ScaPassDead',
 'ScaDrib',
 'ScaSh',
 'ScaFld',
 'ScaDef',
 'GCA',
 'GcaPassLive',
 'GcaPassDead',
 'GcaDrib',
 'GcaSh',
 'GcaFld',
 'GcaDef',
 'Tkl',
 'TklWon',
 'TklDef3rd',
 'TklMid3rd',
 'TklAtt3rd',
 'TklDri',
 'TklDriAtt',
 'TklDri%',
 'TklDriPast',
 'Blocks',
 'BlkSh',
 'BlkPass',
 'Int',
 'Tkl+Int',
 'Clr',
 'Err',
 'Touches',
 'TouDefPen',
 

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Load your dataset
df_players = pd.read_csv("player_data_with_reddit_sentiment.csv")  # should contain 'Player' column

# Prepare list of unique players
player_list = df_players['Player'].unique()

# Base URL template for injury pages
# Note: Transfermarkt uses player-specific URLs like https://www.transfermarkt.com/{slug}/verletzungen/spieler/{id}
# Since we may not have the slug or id in your dataset, we first need to search the player to get it.
search_url_template = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={}"

# DataFrame to store results
injury_results = []

# Loop through players
for i, player in enumerate(player_list, start=1):
    print(f"Processing player {i}/{len(player_list)}: {player}")

    try:
        # 1️⃣ Search the player to get their profile URL
        search_url = search_url_template.format(player.replace(" ", "+"))
        response = requests.get(search_url, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(response.text, "html.parser")

        # Try to find the first search result
        link_tag = soup.select_one("a.spielprofil_tooltip")
        if link_tag:
            player_href = link_tag['href']
            player_slug_id = player_href.split("/spieler/")[-1]  # e.g., '406499'
            player_slug = player_href.split("/")[1]

            # 2️⃣ Build injury page URL
            injury_url = f"https://www.transfermarkt.com/{player_slug}/verletzungen/spieler/{player_slug_id}"
            injury_resp = requests.get(injury_url, headers={"User-Agent": "Mozilla/5.0"})
            injury_soup = BeautifulSoup(injury_resp.text, "html.parser")

            # 3️⃣ Find the injuries table
            table = injury_soup.find("table", class_="items")
            if table:
                rows = table.find_all("tr", class_=["odd", "even"])
                for r in rows:
                    cols = r.find_all("td")
                    if len(cols) >= 5:
                        injury_type = cols[1].text.strip()
                        start_date = cols[2].text.strip()
                        end_date = cols[3].text.strip()
                        days_out = cols[4].text.strip()

                        injury_results.append({
                            "Player": player,
                            "injury_type": injury_type,
                            "start_date": start_date,
                            "end_date": end_date,
                            "days_out": days_out
                        })
            else:
                # No injuries found
                injury_results.append({
                    "Player": player,
                    "injury_type": "None",
                    "start_date": "",
                    "end_date": "",
                    "days_out": 0
                })
        else:
            # Player not found
            injury_results.append({
                "Player": player,
                "injury_type": "Not Found",
                "start_date": "",
                "end_date": "",
                "days_out": 0
            })

    except Exception as e:
        injury_results.append({
            "Player": player,
            "injury_type": "Error",
            "start_date": "",
            "end_date": "",
            "days_out": str(e)
        })

    # Pause to avoid IP ban
    time.sleep(2)

# Convert results to DataFrame
df_injuries = pd.DataFrame(injury_results)

# Save to CSV
df_injuries.to_csv("player_injuries.csv", index=False)
print("✅ Injury scraping completed and saved to player_injuries.csv")


Processing player 1/2266: Brenden Aaronson
Processing player 2/2266: Yunis Abdelhamid
Processing player 3/2266: Himad Abdelli
Processing player 4/2266: Salis Abdul Samed
Processing player 5/2266: Laurent Abergel
Processing player 6/2266: Oliver Abildgaard
Processing player 7/2266: Matthis Abline
Processing player 8/2266: Abner
Processing player 9/2266: Zakaria Aboukhlal
Processing player 10/2266: Tammy Abraham
Processing player 11/2266: Francesco Acerbi
Processing player 12/2266: Mohamed Achi
Processing player 13/2266: Marcos Acuña
Processing player 14/2266: Che Adams
Processing player 15/2266: Tyler Adams
Processing player 16/2266: Sargis Adamyan
Processing player 17/2266: Tosin Adarabioyo
Processing player 18/2266: Martin Adeline
Processing player 19/2266: Karim Adeyemi
Processing player 20/2266: Amine Adli
Processing player 21/2266: Yacine Adli
Processing player 22/2266: Michel Aebischer
Processing player 23/2266: Felix Afena-Gyan
Processing player 24/2266: Emmanuel Agbadou
Processi

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# Load your player list (from your main dataset)
df = pd.read_csv("2022-2023 Football Player Stats.csv", delimiter=';', encoding='latin1')  # Replace with your actual dataset
player_list = df['Player'].dropna().unique()

base_search_url = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query="
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

injury_results = []

for i, player in enumerate(player_list, start=1):
    print(f"🔍 [{i}/{len(player_list)}] Searching for: {player}")
    try:
        # Step 1️⃣: Find player profile
        search_url = base_search_url + player.replace(" ", "+")
        response = requests.get(search_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Find valid player profile
        link_tag = None
        for a in soup.find_all("a", href=True):
            if "/profil/spieler/" in a["href"]:
                link_tag = a
                break

        if not link_tag:
            print(f"⚠️ No Transfermarkt profile found for {player}")
            injury_results.append({
                "Player": player,
                "Injury": "Not Found",
                "From": "",
                "Until": "",
                "Days": ""
            })
            continue

        href = link_tag["href"]
        slug = href.split("/")[1]
        player_id = href.split("/spieler/")[-1]

        # Step 2️⃣: Go to injury page
        injury_url = f"https://www.transfermarkt.com/{slug}/verletzungen/spieler/{player_id}"
        injury_resp = requests.get(injury_url, headers=headers, timeout=10)
        injury_soup = BeautifulSoup(injury_resp.text, "html.parser")

        # Step 3️⃣: Locate injury table
        table = injury_soup.find("table", class_="items")

        if not table:
            print(f"ℹ️ No injury data for {player}")
            injury_results.append({
                "Player": player,
                "Injury": "None",
                "From": "",
                "Until": "",
                "Days": ""
            })
            continue

        # Step 4️⃣: Parse injury table
        rows = table.find_all("tr", class_=["odd", "even"])
        if not rows:
            injury_results.append({
                "Player": player,
                "Injury": "None",
                "From": "",
                "Until": "",
                "Days": ""
            })
            continue

        for row in rows:
            cols = [td.text.strip() for td in row.find_all("td")]
            if len(cols) >= 5:
                injury_results.append({
                    "Player": player,
                    "Injury": cols[1],
                    "From": cols[2],
                    "Until": cols[3],
                    "Days": cols[4]
                })

        # Random delay (2–5 seconds)
        time.sleep(random.uniform(2, 5))

    except Exception as e:
        print(f"❗ Error scraping {player}: {e}")
        injury_results.append({
            "Player": player,
            "Injury": "Error",
            "From": "",
            "Until": "",
            "Days": str(e)
        })

# Step 5️⃣: Save all results
df_injuries = pd.DataFrame(injury_results)
df_injuries.to_csv("player_injuries_all.csv", index=False)
print("\n✅ Scraping completed successfully! File saved as player_injuries_all.csv")

🔍 [1/2530] Searching for: Brenden Aaronson
🔍 [2/2530] Searching for: Yunis Abdelhamid
🔍 [3/2530] Searching for: Himad Abdelli
🔍 [4/2530] Searching for: Salis Abdul Samed
🔍 [5/2530] Searching for: Laurent Abergel
🔍 [6/2530] Searching for: Oliver Abildgaard
🔍 [7/2530] Searching for: Matthis Abline
ℹ️ No injury data for Matthis Abline
🔍 [8/2530] Searching for: Abner
🔍 [9/2530] Searching for: Zakaria Aboukhlal
🔍 [10/2530] Searching for: Tammy Abraham
🔍 [11/2530] Searching for: Francesco Acerbi
🔍 [12/2530] Searching for: Mohamed Achi
ℹ️ No injury data for Mohamed Achi
🔍 [13/2530] Searching for: Marcos Acuña
🔍 [14/2530] Searching for: Che Adams
🔍 [15/2530] Searching for: Tyler Adams
🔍 [16/2530] Searching for: Sargis Adamyan
🔍 [17/2530] Searching for: Tosin Adarabioyo
🔍 [18/2530] Searching for: Martin Adeline
🔍 [19/2530] Searching for: Karim Adeyemi
🔍 [20/2530] Searching for: Amine Adli
🔍 [21/2530] Searching for: Yacine Adli
🔍 [22/2530] Searching for: Michel Aebischer
🔍 [23/2530] Searching fo

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time, random

# ===============================================
# 1️⃣ Load dataset and filter failed players
# ===============================================
df = pd.read_csv("player_injuries_all.csv")

error_mask = df['Injury'].astype(str).str.contains(
    "503|403|404|Error|timeout|Service Unavailable|Read timed out|No Profile|Failed",
    case=False, na=False
)
failed_players = df.loc[error_mask, 'Player'].dropna().unique().tolist()
print(f"🧩 Retrying {len(failed_players)} players with previous errors...")

# ===============================================
# 2️⃣ Helper: scrape injury data safely
# ===============================================
def get_injury_data(player_name, retries=3):
    headers = {"User-Agent": "Mozilla/5.0"}
    search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={player_name.replace(' ', '+')}"

    for attempt in range(retries):
        try:
            search_resp = requests.get(search_url, headers=headers, timeout=15)
            search_resp.raise_for_status()
            soup = BeautifulSoup(search_resp.text, "html.parser")

            # 🟩 Find the correct table containing player results
            table = soup.find("table", class_="items")
            if not table:
                return pd.DataFrame([{"Player": player_name, "Injury": "No Profile Found"}])

            # 🟩 Extract the first valid player link
            link_tag = None
            for a in table.find_all("a", href=True):
                if "/profil/spieler/" in a["href"]:
                    link_tag = a
                    break

            if not link_tag:
                return pd.DataFrame([{"Player": player_name, "Injury": "No Profile Found"}])

            profile_link = link_tag["href"]
            # Extract player_id safely
            try:
                player_id = profile_link.split("/")[-1]
                slug = profile_link.split("/profil/")[0]
            except Exception:
                return pd.DataFrame([{"Player": player_name, "Injury": "Malformed profile link"}])

            injury_url = f"https://www.transfermarkt.com{slug}/verletzungen/spieler/{player_id}"

            injury_resp = requests.get(injury_url, headers=headers, timeout=15)
            injury_resp.raise_for_status()
            soup_injury = BeautifulSoup(injury_resp.text, "html.parser")

            # 🩺 Parse the injury table
            table = soup_injury.find("table", class_="items")
            if not table:
                return pd.DataFrame([{"Player": player_name, "Injury": "No Injury Data"}])

            rows = table.find_all("tr", {"class": ["odd", "even"]})
            injury_data = []
            for row in rows:
                cols = [c.text.strip() for c in row.find_all("td")]
                if len(cols) >= 5:
                    injury_data.append({
                        "Player": player_name,
                        "Injury": cols[1],
                        "From": cols[2],
                        "Until": cols[3],
                        "Days": cols[4]
                    })

            if not injury_data:
                return pd.DataFrame([{"Player": player_name, "Injury": "No Injury Data"}])

            return pd.DataFrame(injury_data)

        except requests.exceptions.Timeout:
            print(f"⏳ Timeout for {player_name}, retry {attempt+1}/{retries}...")
            time.sleep(random.uniform(8, 15))
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request error for {player_name}: {e}")
            if attempt < retries - 1:
                time.sleep(random.uniform(8, 15))
                continue
            return pd.DataFrame([{"Player": player_name, "Injury": f"Error: {str(e)}"}])

    return pd.DataFrame([{"Player": player_name, "Injury": "Failed after retries"}])

# ===============================================
# 3️⃣ Retry only failed players
# ===============================================
retry_results = []
for i, player in enumerate(failed_players, start=1):
    print(f"🔁 [{i}/{len(failed_players)}] Retrying: {player}")
    df_player = get_injury_data(player)
    retry_results.append(df_player)
    time.sleep(random.uniform(6, 12))  # polite delay

# ===============================================
# 4️⃣ Merge new results with old dataset
# ===============================================
df_retry = pd.concat(retry_results, ignore_index=True)
df_cleaned = df[~df['Player'].isin(failed_players)]
df_final = pd.concat([df_cleaned, df_retry], ignore_index=True).drop_duplicates(subset=["Player", "Injury", "From"], keep="last")
df_final.to_csv("player_injuries_final_retry_fixed.csv", index=False)
print("✅ Retry complete! Fixed file saved as: player_injuries_final_retry_fixed.csv")


🧩 Retrying 151 players with previous errors...
🔁 [1/151] Retrying: Lorenz Assignon
🔁 [2/151] Retrying: Youcef Atal
🔁 [3/151] Retrying: Pierre-Emerick Aubameyang
🔁 [4/151] Retrying: Emil Audero
🔁 [5/151] Retrying: Tommaso Augello
🔁 [6/151] Retrying: Ludwig Augustinsson
🔁 [7/151] Retrying: Serge Aurier
🔁 [8/151] Retrying: Mathias Autret
🔁 [9/151] Retrying: Ezequiel Ávila
🔁 [10/151] Retrying: Cédric Avinel
🔁 [11/151] Retrying: Taiwo Awoniyi
🔁 [12/151] Retrying: Mehmet-Can Aydin
🔁 [13/151] Retrying: André Ayew
🔁 [14/151] Retrying: Jordan Ayew
🔁 [15/151] Retrying: Kaan Ayhan
🔁 [16/151] Retrying: Luke Ayling
🔁 [17/151] Retrying: Ayman Azhil
🔁 [18/151] Retrying: Sardar Azmoun
🔁 [19/151] Retrying: César Azpilicueta
⏳ Timeout for César Azpilicueta, retry 1/3...
🔁 [20/151] Retrying: Sanoussy Ba
🔁 [21/151] Retrying: Iddrisu Baba
🔁 [22/151] Retrying: Sr?an Babi?
🔁 [23/151] Retrying: Loïc Bade
🔁 [24/151] Retrying: Édgar Badía
🔁 [25/151] Retrying: Benoît Badiashile
🔁 [26/151] Retrying: Alex Baena
🔁 

In [ ]:
import pandas as pd
from datetime import datetime

# Load your full injury dataset
df_inj = pd.read_csv("player_injuries_final_retry_fixed.csv")

# Clean "Days" column (convert "36 days" → 36)
df_inj["Days_clean"] = (
    df_inj["Days"]
    .fillna("0")
    .str.replace(" days", "")
    .str.replace(" day", "")
    .replace("-", "0")
    .astype(int)
)

# Convert "From" and "Until" columns to datetime (if available)
for col in ["From", "Until"]:
    df_inj[col] = pd.to_datetime(df_inj[col], errors="coerce", dayfirst=True)

# Compute aggregated injury features per player
df_summary = df_inj.groupby("Player").agg(
    total_injuries=("Injury", "count"),
    total_days_injured=("Days_clean", "sum"),
    avg_days_injured=("Days_clean", "mean"),
    max_days_injured=("Days_clean", "max"),
    recent_injury_days=("Days_clean", "first")  # optional — most recent injury
).reset_index()

# Fill missing values with 0
df_summary = df_summary.fillna(0)

# Save aggregated injury summary
df_summary.to_csv("player_injury_summary.csv", index=False)

print("✅ Injury summary saved as player_injury_summary.csv")


✅ Injury summary saved as player_injury_summary.csv


In [ ]:
 import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 🧍 Choose one player name
player = "Brenden Aaronson"   # 🔁 change this name to test another player

base_search_url = "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query="
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

try:
    print(f"🔍 Searching for player: {player}")
    search_url = base_search_url + player.replace(" ", "+")
    response = requests.get(search_url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Find player profile link
    link_tag = None
    for a in soup.find_all("a", href=True):
        if "/profil/spieler/" in a["href"]:
            link_tag = a
            break

    if not link_tag:
        print(f"⚠️ No Transfermarkt profile found for {player}")
    else:
        href = link_tag["href"]
        slug = href.split("/")[1]
        player_id = href.split("/spieler/")[-1]
        injury_url = f"https://www.transfermarkt.com/{slug}/verletzungen/spieler/{player_id}"
        print(f"📄 Player injury page: {injury_url}")

        injury_resp = requests.get(injury_url, headers=headers)
        injury_soup = BeautifulSoup(injury_resp.text, "html.parser")

        # Locate injury table
        table = injury_soup.find("table", class_="items")

        if not table:
            print(f"❌ No injury data found for {player}")
        else:
            # Parse rows
            rows = table.find_all("tr", class_=["odd", "even"])
            injury_data = []
            for row in rows:
                cols = [td.text.strip() for td in row.find_all("td")]
                if len(cols) >= 5:
                    injury_data.append({
                        "Player": player,
                        "Injury": cols[1],
                        "From": cols[2],
                        "Until": cols[3],
                        "Days": cols[4]
                    })

            # Convert to DataFrame
            df_inj = pd.DataFrame(injury_data)
            print("\n✅ Injury Data Extracted:")
            print(df_inj)

except Exception as e:
    print(f"❗ Error: {e}")


🔍 Searching for player: Brenden Aaronson
📄 Player injury page: https://www.transfermarkt.com/brenden-aaronson/verletzungen/spieler/393323

✅ Injury Data Extracted:
             Player         Injury        From       Until     Days
0  Brenden Aaronson            Ill  24/11/2023  27/11/2023   4 days
1  Brenden Aaronson  Knee problems  19/03/2022  23/04/2022  36 days


DATA CLEANING

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("/content/2022-2023 Football Player Stats.csv", delimiter=';', encoding='latin1')

# Overview
df.info()
df.head()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2689 entries, 0 to 2688
Columns: 124 entries, Rk to AerWon%
dtypes: float64(112), int64(7), object(5)
memory usage: 2.5+ MB


,Rk,Age,Born,MP,Starts,Min,90s,Goals,Shots,SoT,...,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%
count,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,...,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000,2689.000000
mean,1345.000000,26.011157,1996.155820,11.833023,8.476013,760.451097,8.450465,1.027520,1.245787,0.411261,...,0.204697,1.661636,0.990569,0.009249,0.015173,0.003142,4.951967,1.312064,1.497356,43.583600
std,776.391761,4.446259,4.450108,6.864278,6.994383,591.094260,6.567484,2.013714,1.424619,0.754716,...,0.552376,2.319000,1.235965,0.043781,0.077399,0.022607,2.901833,1.579539,1.830391,26.673092
min,1.000000,15.000000,1981.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,673.000000,23.000000,1993.000000,5.000000,2.000000,194.000000,2.200000,0.000000,0.260000,0.000000,...,0.000000,0.000000,0.300000,0.000000,0.000000,0.000000,3.330000,0.330000,0.550000,27.600000
50%,1345.000000,26.000000,1996.000000,13.000000,7.000000,684.000000,7.600000,0.000000,0.860000,0.180000,...,0.000000,0.760000,0.830000,0.000000,0.000000,0.000000,5.000000,0.930000,1.100000,46.400000
75%,2017.000000,29.000000,2000.000000,18.000000,14.000000,1245.000000,13.800000,1.000000,1.850000,0.590000,...,0.210000,2.500000,1.320000,0.000000,0.000000,0.000000,6.270000,1.790000,1.830000,60.000000
max,2689.000000,41.000000,2007.000000,23.000000,23.000000,2070.000000,23.000000,25.000000,15.000000,10.000000,...,10.000000,30.000000,20.000000,0.870000,2.000000,0.500000,30.000000,25.000000,30.000000,100.000000


In [ ]:
df.drop(columns=['Rk'], inplace=True, errors='ignore')

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(2689, 123)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

,0
Nation,1
Player,0
Pos,0
Squad,0
Comp,0
...,...
OG,0
Recov,0
AerWon,0
AerLost,0


In [ ]:
df['Nation'] = df['Nation'].fillna(df['Nation'].mode()[0])

In [ ]:
df.isnull().sum().sort_values(ascending=False)

,0
Player,0
Nation,0
Pos,0
Squad,0
Comp,0
...,...
OG,0
Recov,0
AerWon,0
AerLost,0


In [ ]:
#5️ Convert Data Types
numeric_cols = df.columns.drop(['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Born'])
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
#6 Clean Percentage Columns
percent_cols = [col for col in df.columns if df[col].astype(str).str.contains('%').any()]
for col in percent_cols:
    df[col] = df[col].astype(str).str.replace('%','').astype(float)

In [ ]:
#7 Normalize Column Names
df.columns = (df.columns
                .str.strip()
                .str.replace(' ', '_')
                .str.replace('%', 'pct')
                .str.replace('+', '_plus_')
                .str.lower())

In [ ]:
#8 Fix Categorical Fields
for col in ['player', 'nation', 'pos', 'squad', 'comp']:
    df[col] = df[col].str.strip().str.title()

In [ ]:
#9 Feature Engineering (Optional)
df['goals_per90'] = df['goals'] / df['90s']
df['assists_per90'] = df['assists'] / df['90s']
df['shots_on_target_rate'] = df['sot'] / df['shots']

/tmp/ipython-input-4017916828.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['goals_per90'] = df['goals'] / df['90s']
/tmp/ipython-input-4017916828.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['assists_per90'] = df['assists'] / df['90s']
/tmp/ipython-input-4017916828.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fr

In [ ]:
df = df.copy()

In [ ]:
#10 Outlier Detection
df = df[(df['sot'] <= 100) & (df['age'] < 45)]

In [ ]:
df.shape

(2689, 126)

In [ ]:
import numpy as np

# 1. Replace infinities with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Fill numeric NaN with 0
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(0)

# 3. Fill text NaN with 'Unknown'
str_cols = df.select_dtypes(include=['object']).columns
df[str_cols] = df[str_cols].fillna('Unknown')

# 4. Verify
print(df.isna().sum().sort_values(ascending=False).head(10))


player    0
nation    0
pos       0
squad     0
comp      0
age       0
born      0
mp        0
starts    0
min       0
dtype: int64


In [ ]:
df.isnull().sum().sort_values(ascending=False)

,0
player,0
nation,0
pos,0
squad,0
comp,0
...,...
aerlost,0
aerwonpct,0
goals_per90,0
assists_per90,0


In [ ]:
df.to_csv("clean_player_performance.csv", index=False)

In [ ]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
from unidecode import unidecode

# ======================================
# 1️⃣ Load the two datasets
# ======================================
main_df = pd.read_csv("player_data_with_reddit_sentiment.csv")   # your main dataset
secondary_df = pd.read_csv("player_injury_summary.csv")          # dataset to merge (e.g., injury)

# ======================================
# 2️⃣ Define the cleaning function
# ======================================
def clean_text(name):
    if pd.isna(name):
        return ""
    name = unidecode(str(name).lower().strip())
    name = re.sub(r'[^a-z\s]', '', name)  # keep only letters and spaces
    name = re.sub(r'\s+', ' ', name)      # collapse multiple spaces
    return name.title()

# ======================================
# 3️⃣ Clean the player names in both datasets
# ======================================
main_df["player_clean"] = main_df["Player"].apply(clean_text)
secondary_df["player_clean"] = secondary_df["Player"].apply(clean_text)

# ======================================
# 4️⃣ Merge on the cleaned player name
# ======================================
merged = main_df.merge(secondary_df, on="player_clean", how="left")

# ======================================
# 5️⃣ Save the merged dataset
# ======================================
merged.to_csv("merged_player_data.csv", index=False)

# ======================================
# 6️⃣ Verify result
# ======================================
print("✅ Merge completed successfully!")
print("Main dataset shape:", main_df.shape)
print("Secondary dataset shape:", secondary_df.shape)
print("Merged dataset shape:", merged.shape)
print("\nSample merged data:")
print(merged[["Player_x", "Player_y", "player_clean"]].head(10))


✅ Merge completed successfully!
Main dataset shape: (2415, 135)
Secondary dataset shape: (2530, 7)
Merged dataset shape: (2417, 141)

Sample merged data:
            Player_x           Player_y       player_clean
0   Brenden Aaronson   Brenden Aaronson   Brenden Aaronson
1   Yunis Abdelhamid   Yunis Abdelhamid   Yunis Abdelhamid
2      Himad Abdelli      Himad Abdelli      Himad Abdelli
3  Salis Abdul Samed  Salis Abdul Samed  Salis Abdul Samed
4    Laurent Abergel    Laurent Abergel    Laurent Abergel
5  Oliver Abildgaard  Oliver Abildgaard  Oliver Abildgaard
6     Matthis Abline     Matthis Abline     Matthis Abline
7     Matthis Abline     Matthis Abline     Matthis Abline
8              Abner              Abner              Abner
9  Zakaria Aboukhlal  Zakaria Aboukhlal  Zakaria Aboukhlal


In [ ]:
import pandas as pd

# Load your merged dataset
merged = pd.read_csv("merged_player_data.csv")

# Drop redundant/duplicate columns
columns_to_drop = [
    "Player_x", "Player_y", "Player",
    "Market Value", "Market Value_x", "Market Value_y",
    "Market Value Euros_x", "Market Value Euros_y",
    "PasCmp", "PasAtt", "Crs", "TklW"
]

merged.drop(columns=columns_to_drop, errors="ignore", inplace=True)

# Keep only cleaned player name for reference
if "player_clean" in merged.columns:
    merged.rename(columns={"player_clean": "player_id"}, inplace=True)

# Save ML-ready dataset
merged.to_csv("player_data_ml_ready.csv", index=False)

print("✅ ML-ready dataset saved as 'player_data_ml_ready.csv'")
print("Final shape:", merged.shape)


✅ ML-ready dataset saved as 'player_data_ml_ready.csv'
Final shape: (2417, 130)


In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("player_data_ml_ready.csv")

# 1️⃣ Drop unnecessary and duplicate rows
df.drop(columns=['Rk'], inplace=True, errors='ignore')
df.drop_duplicates(inplace=True)

# 2️⃣ Fill missing values
if 'Nation' in df.columns:
    df['Nation'] = df['Nation'].fillna(df['Nation'].mode()[0])

# 3️⃣ Convert numeric columns
non_numeric = ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Born']
numeric_cols = [c for c in df.columns if c not in non_numeric]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 4️⃣ Handle percentage-like columns safely
percent_cols = [c for c in df.columns if df[c].dtype == 'object' and df[c].astype(str).str.contains('%').any()]
for col in percent_cols:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5️⃣ Normalize column names
df.columns = (df.columns
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('%', 'pct')
              .str.replace('+', '_plus_')
              .str.lower())

# 6️⃣ Standardize categorical columns
for col in ['player', 'nation', 'pos', 'squad', 'comp']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

# 7️⃣ Feature Engineering (guard against division by zero)
if '90s' in df.columns:
    df['goals_per90'] = df['goals'] / df['90s'].replace(0, np.nan)
    df['assists_per90'] = df['assists'] / df['90s'].replace(0, np.nan)
    df['shots_on_target_rate'] = df['sot'] / df['shots'].replace(0, np.nan)

# 8️⃣ Handle infinities, NaNs, and outliers
df.replace([np.inf, -np.inf], np.nan, inplace=True)
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(0)
str_cols = df.select_dtypes(include=['object']).columns
df[str_cols] = df[str_cols].fillna('unknown')

# Optional: Outlier removal using quantiles instead of fixed values
if 'sot' in df.columns and 'age' in df.columns:
    df = df[df['sot'] <= df['sot'].quantile(0.99)]
    df = df[df['age'] <= df['age'].quantile(0.99)]

# 9️⃣ Save cleaned dataset
df.to_csv("final_dataset.csv", index=False)
print("✅ Cleaned dataset saved as 'clean_player_performance.csv'")
print("Final shape:", df.shape)
print(df.isna().sum().sort_values(ascending=False).head(10))


/tmp/ipython-input-241765660.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['goals_per90'] = df['goals'] / df['90s'].replace(0, np.nan)
/tmp/ipython-input-241765660.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['assists_per90'] = df['assists'] / df['90s'].replace(0, np.nan)
/tmp/ipython-input-241765660.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis

✅ Cleaned dataset saved as 'clean_player_performance.csv'
Final shape: (2380, 132)
nation    0
pos       0
squad     0
comp      0
age       0
born      0
mp        0
starts    0
min       0
90s       0
dtype: int64


In [ ]:
# ===============================================================
# ⚙️ PLAYER MARKET VALUE FEATURE ENGINEERING PIPELINE
# ===============================================================
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("final_dataset.csv")

# ---------------------------------------------------------------
# 🧹 1. Column normalization and cleanup
# ---------------------------------------------------------------
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.copy()

# Handle missing / invalid entries
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# ---------------------------------------------------------------
# 🧩 2. Basic Efficiency Metrics
# ---------------------------------------------------------------
if 'goals' in df.columns and '90s' in df.columns:
    df['goals_per90'] = df['goals'] / df['90s'].replace(0, np.nan)
if 'assists' in df.columns and '90s' in df.columns:
    df['assists_per90'] = df['assists'] / df['90s'].replace(0, np.nan)
if {'goals', 'assists', '90s'}.issubset(df.columns):
    df['goal_contrib_per90'] = (df['goals'] + df['assists']) / df['90s'].replace(0, np.nan)

# ---------------------------------------------------------------
# 🎯 3. Shooting & Passing
# ---------------------------------------------------------------
if {'sot', 'shots'}.issubset(df.columns):
    df['shots_on_target_rate'] = df['sot'] / df['shots'].replace(0, np.nan)
if {'pastotcmp', 'pastotatt'}.issubset(df.columns):
    df['pass_accuracy'] = df['pastotcmp'] / df['pastotatt'].replace(0, np.nan)
if {'pasprog', 'pastotatt'}.issubset(df.columns):
    df['progressive_pass_rate'] = df['pasprog'] / df['pastotatt'].replace(0, np.nan)

# ---------------------------------------------------------------
# 🛡️ 4. Defensive & Physical Features
# ---------------------------------------------------------------
defense_cols = [c for c in ['tkl', 'int', 'blocks'] if c in df.columns]
if defense_cols:
    df['defensive_actions'] = df[defense_cols].sum(axis=1)

if {'crdy', 'crdr'}.issubset(df.columns):
    df['discipline_score'] = df['crdy'] + (2 * df['crdr'])

if {'stamina', 'strength', 'jumping'}.issubset(df.columns):
    df['physical_strength'] = (df['stamina'] + df['strength'] + df['jumping']) / 3

# ---------------------------------------------------------------
# 🩺 5. Injury-Related Features
# ---------------------------------------------------------------
if {'total_injuries', 'total_days_injured'}.issubset(df.columns):
    df['avg_days_per_injury'] = df['total_days_injured'] / df['total_injuries'].replace(0, np.nan)
    df['injury_ratio'] = df['total_injuries'] / (df['90s'] + 1)

# ---------------------------------------------------------------
# 💬 6. Sentiment-Based Features
# ---------------------------------------------------------------
if {'positive', 'neutral', 'negative'}.issubset(df.columns):
    df['net_sentiment'] = df['positive'] - df['negative']
    df['sentiment_ratio'] = (df['positive'] + 1) / (df['negative'] + 1)

# ---------------------------------------------------------------
# 📊 7. Composite Feature Scores (inspired by notebook logic)
# ---------------------------------------------------------------
if {'goals_per90', 'assists_per90', 'pass_accuracy', 'shots_on_target_rate'}.issubset(df.columns):
    df['offensive_index'] = (
        0.4 * df['goals_per90'] +
        0.3 * df['assists_per90'] +
        0.2 * df['pass_accuracy'] +
        0.1 * df['shots_on_target_rate']
    )

if {'defensive_actions', 'discipline_score'}.issubset(df.columns):
    df['defensive_index'] = df['defensive_actions'] / (df['discipline_score'] + 1)

if {'physical_strength', 'defensive_index', 'offensive_index'}.issubset(df.columns):
    df['total_player_index'] = (
        0.5 * df['offensive_index'] +
        0.3 * df['defensive_index'] +
        0.2 * df['physical_strength']
    )

# ---------------------------------------------------------------
# 🔍 8. Normalize numeric columns (optional scaling)
# ---------------------------------------------------------------
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].apply(lambda x: (x - x.mean()) / (x.std() + 1e-8))

# ---------------------------------------------------------------
# 💾 9. Save engineered dataset
# ---------------------------------------------------------------
output_path = "final_dataset_feature_engineered.csv"
df.to_csv(output_path, index=False)

print("✅ Feature Engineering Completed Successfully!")
print("Saved as:", output_path)
print("Total features:", df.shape[1])

# Show sample engineered features
feature_cols = [
    'goals_per90', 'assists_per90', 'goal_contrib_per90',
    'shots_on_target_rate', 'pass_accuracy', 'progressive_pass_rate',
    'defensive_actions', 'discipline_score', 'physical_strength',
    'avg_days_per_injury', 'net_sentiment', 'sentiment_ratio',
    'offensive_index', 'defensive_index', 'total_player_index'
]
available = [c for c in feature_cols if c in df.columns]
print("\nEngineered Features:", available)
df[available].head(10)


✅ Feature Engineering Completed Successfully!
Saved as: final_dataset_feature_engineered.csv
Total features: 143

Engineered Features: ['goals_per90', 'assists_per90', 'goal_contrib_per90', 'shots_on_target_rate', 'pass_accuracy', 'progressive_pass_rate', 'defensive_actions', 'discipline_score', 'avg_days_per_injury', 'net_sentiment', 'sentiment_ratio', 'offensive_index', 'defensive_index']


,goals_per90,assists_per90,goal_contrib_per90,shots_on_target_rate,pass_accuracy,progressive_pass_rate,defensive_actions,discipline_score,avg_days_per_injury,net_sentiment,sentiment_ratio,offensive_index,defensive_index
0,-0.260489,-0.049499,-0.137618,-0.493886,-0.116812,0.418562,-0.078344,-0.205342,-0.501406,1.365442,1.446933,-0.441299,0.030070
1,-0.532791,-0.056544,-0.239372,-1.042824,0.484004,0.281825,1.201662,-0.229504,-0.737780,-1.183204,-1.096227,-0.637410,1.350596
2,-0.532791,-0.060606,-0.243199,0.166977,0.416171,0.825985,0.672797,-0.193261,-0.678687,-2.455482,-1.892317,-0.419735,0.759148
3,-0.291804,-0.060606,-0.159020,-0.199371,1.386841,0.136748,0.117106,-0.157019,0.732172,-0.870940,-0.864855,-0.132495,0.172798
4,-0.532791,-0.049522,-0.232758,-1.298415,0.998239,0.615015,1.059865,-0.144938,-0.506330,-1.124193,-1.042073,-0.578229,1.057923
5,-0.532791,-0.060606,-0.243199,NaN,-0.846464,-1.246117,2.454842,-0.338230,-0.261749,-1.474809,-1.307235,NaN,2.969669
6,1.762320,-0.060606,0.558506,0.166977,-0.423842,1.798902,-0.461580,-0.338230,NaN,-0.398139,-0.502390,1.056122,-0.250965
7,3.483654,-0.060606,1.159784,0.169908,-2.634571,0.895012,-0.101338,-0.338230,NaN,-0.398139,-0.502390,1.859167,0.146853
8,-0.532791,-0.060606,-0.243199,0.866866,-0.386055,-0.263281,0.921900,-0.338230,-0.402917,-0.305732,-0.402188,-0.437355,1.276826
9,0.910243,-0.034921,0.285062,0.332167,-0.587663,-0.105898,-0.411759,-0.120777,0.217566,-0.758463,-0.840658,0.485963,-0.358633


SENTIMENT ANALYSIS

In [ ]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ===============================
# 1️⃣ Load the Reddit Sentiment Data
# ===============================
df = pd.read_csv("reddit_sentiment_data.csv")

print("Before cleaning:", df.shape)
df.head()

# ===============================
# 2️⃣ Basic Cleaning
# ===============================

# Drop rows with missing or empty text
df = df[df["Text"].notna()]
df = df[df["Text"].str.strip() != ""]

# Remove duplicates
df = df.drop_duplicates(subset=["Player", "Text"])

# Remove extremely short or meaningless posts
df = df[df["Text"].str.len() > 20]

print("After cleaning:", df.shape)

# ===============================
# 3️⃣ Recompute Sentiment (Optional)
# ===============================
# In case the earlier sentiment scores were noisy or missing
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(text)["compound"]
    if score > 0.05:
        return "positive", score
    elif score < -0.05:
        return "negative", score
    else:
        return "neutral", score

df[["Sentiment", "SentimentScore"]] = df["Text"].apply(
    lambda t: pd.Series(get_sentiment(t))
)

# ===============================
# 4️⃣ Aggregate Sentiment Per Player
# ===============================
sentiment_summary = (
    df.groupby("Player")["Sentiment"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

sentiment_summary["AvgSentimentScore"] = df.groupby("Player")["SentimentScore"].mean()
sentiment_summary.reset_index(inplace=True)

print("✅ Sentiment summary created:")
print(sentiment_summary.head())

# ===============================
# 5️⃣ Save Cleaned + Summary Data
# ===============================
'''df.to_csv("reddit_sentiment_cleaned.csv", index=False)
sentiment_summary.to_csv("reddit_sentiment_summary.csv", index=False)

print("✅ Cleaned data saved as reddit_sentiment_cleaned.csv")
print("✅ Sentiment summary saved as reddit_sentiment_summary.csv")'''


Before cleaning: (143950, 4)
After cleaning: (143856, 4)
✅ Sentiment summary created:
Sentiment                 Player  negative   neutral  positive  \
0          ?órir Jóhann Helgason  0.470588  0.000000  0.529412   
1                Aaron Cresswell  0.102564  0.102564  0.794872   
2                   Aaron Hickey  0.370370  0.185185  0.444444   
3                 Aaron Ramsdale  0.410000  0.170000  0.420000   
4                   Aaron Ramsey  0.240000  0.350000  0.410000   

Sentiment  AvgSentimentScore  
0                   0.125700  
1                   0.624788  
2                   0.023639  
3                   0.003236  
4                   0.105102  


'df.to_csv("reddit_sentiment_cleaned.csv", index=False)\nsentiment_summary.to_csv("reddit_sentiment_summary.csv", index=False)\n\nprint("✅ Cleaned data saved as reddit_sentiment_cleaned.csv")\nprint("✅ Sentiment summary saved as reddit_sentiment_summary.csv")'

In [ ]:
df.to_csv("reddit_sentiment_cleaned.csv", index=False)
sentiment_summary.to_csv("reddit_sentiment_summary.csv", index=False)

print("✅ Cleaned data saved as reddit_sentiment_cleaned.csv")
print("✅ Sentiment summary saved as reddit_sentiment_summary.csv")

✅ Cleaned data saved as reddit_sentiment_cleaned.csv
✅ Sentiment summary saved as reddit_sentiment_summary.csv
